In [1]:
#cellule technique
! pip install yfinance pandas

import yfinance as yf
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np

def get_sp500_returns():
    # 1. Récupération de la liste des tickers du S&P 500 via Wikipédia
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

    # Ajout d'un User-Agent pour éviter le blocage 403 Forbidden
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Lève une exception pour les codes d'état d'erreur

    table = pd.read_html(response.text)
    df_tickers = table[0]
    tickers = df_tickers['Symbol'].tolist()

    # Correction pour les tickers contenant des points (ex: BRK.B -> BRK-B pour Yahoo Finance)
    tickers = [t.replace('.', '-') for t in tickers]

    print(f"Téléchargement des données pour {len(tickers)} entreprises...")
    return (tickers)

liste= get_sp500_returns()
tickers = liste[0:20]

Téléchargement des données pour 503 entreprises...


/tmp/ipykernel_1412/2670877908.py:19: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(response.text)


In [2]:
# fonction pour visualiser
def visualise(ticker):
  fig, (ax1, ax2, ax3) = plt.subplots(nrows=3, ncols=1, figsize=(12, 15))

  #courbe des prix
  data= yf.download(ticker, start="2025-01-01", end="2026-01-01")["Close"]

  ax1.plot(data, label='Prix de clôture (MMM)', color='blue')
  ax1.set_title('Évolution du prix de clôture de MMM (2025)')
  ax1.set_xlabel('Date')
  ax1.set_ylabel('Prix de clôture (USD)')
  ax1.grid(True, linestyle='--', alpha=0.7)
  ax1.legend()

  #rendements
  returns= data.pct_change()[1:]
  # Tracer la courbe
  ax2.plot(returns, label='Rendement journalier (MMM)', color='red')
  ax2.set_title('Évolution du rendement de MMM (2025)')
  ax2.set_xlabel('Date')
  ax2.set_ylabel('Rendement en % ')
  ax2.grid(True, linestyle='--', alpha=0.7)
  ax2.legend()

  # Tracer l'histogramme (graph3)
  ax3.hist(returns, bins=30, color='skyblue', edgecolor='black', alpha=0.7, label='Rendements quotidiens')

  # Ajouter des lignes pour la moyenne et l'écart-type
  moyenne = returns.mean()
  ecart_type = returns.std()
  ax3.axvline(moyenne.iloc[0], color='red', linestyle='--', label=f'Moyenne : {moyenne.iloc[0]:.2f}%')
  ax3.axvline(moyenne.iloc[0] + ecart_type.iloc[0], color='green', linestyle=':', label=f'+1σ : {(moyenne + ecart_type).iloc[0]:.2f}%')
  ax3.axvline(moyenne.iloc[0] - ecart_type.iloc[0], color='green', linestyle=':', label=f'-1σ : {(moyenne - ecart_type).iloc[0]:.2f}%')

  #ajouter une gaussienne
  X=np.linspace(moyenne-5*ecart_type,moyenne+5*ecart_type,100)
  def f(x):
    return 1.6/(ecart_type.iloc[0]*np.sqrt(2*np.pi))*np.exp(-(x-moyenne.iloc[0])**2/(2*ecart_type.iloc[0]**2))
  ax3.plot(X,f(X),color='black',label='Gaussienne')

  # Personnalisation
  ax3.set_title('Histogramme des rendements quotidiens de MMM (2025)')
  ax3.set_xlabel('Rendement quotidien (%)')
  ax3.set_ylabel('Fréquence')
  ax3.grid(True, linestyle='--', alpha=0.5)
  ax3.legend()

  #plt.tight_layout() # Adjust layout to prevent overlapping
  plt.show()

In [3]:
# pour obtenir la matrice des rendements et de covariance

def parametre(tickers):
  data= yf.download(tickers, start="2025-01-01", end="2026-01-01")["Close"]
  returns=data.pct_change().dropna()
  r=returns.mean()
  cov=returns.cov()
  return np.array(r*100),np.array(cov*10000)

In [6]:
# permet de trouver le portefeuille optimal

from scipy.linalg import pinv

def inv(A):
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return pinv(A)

#calcul le protefeuille de rendement optimal en fixant la variance a c
def optimal1(R, V, c):
    R = np.asarray(R).flatten()  # Convertir R en vecteur 1D
    V_inv = inv(V)
    P = np.dot(R, np.dot(V_inv, R))  # R^T V^{-1} R
    w = np.dot(V_inv, R) * np.sqrt(c / P)
    return w

#calcul le protefeuille de variance minimale en fixant le rendement a mu
def optimal2(R, V, mu):
    R = np.asarray(R).flatten()  # Convertir R en vecteur 1D
    V_inv = inv(V)
    one=np.ones(len(R))
    a = np.dot(R, np.dot(V_inv,R))
    b = np.dot(one, np.dot(V_inv,R))
    c = np.dot(R, np.dot(V_inv,one))
    d = np.dot(one, np.dot(V_inv,one))
    M=inv([[a,b],[c,d]])
    w=(mu*M[0][0]+M[0][1])*np.dot(R,V_inv)+(mu*M[1][0]+M[1][1])*np.dot(one,V_inv)
    return w


In [10]:
print("le portefeuille opti pour",tickers)
R,V=parametre(tickers)
one=np.ones(len(R))
c=2
w1=optimal1(R,V,c)
mu1=np.dot(w1,R)
s1=np.dot(w1,np.dot(V,w1))
w2=optimal2(R,V,mu1)
mu2=np.dot(w2,R)
s2=np.dot(w2,np.dot(V,w2))
print("rendement(1,2)=",mu1,"-",mu2,"/")
print("variance=",s1,"-",s2,"/")
print("norm1=",np.dot(w1,one),"-",np.dot(w2,one),"/")
print("norm=",np.dot(w1-w2,w1-w2))



/tmp/ipykernel_1412/1412144060.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data= yf.download(tickers, start="2025-01-01", end="2026-01-01")["Close"]
[*****************     35%                       ]  7 of 20 completed

le portefeuille opti pour ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL']


[*********************100%***********************]  20 of 20 completed


rendement(1,2)= 0.3921484727596425 - 0.3921484727596425 /
variance= 2.0000000000000004 - 2.1617546444405846 /
norm1= 0.5500766672221755 - 0.9999999999999994 /
norm= 0.06069823264379792


# **Pourquoi `optimal2` est-il plus adapté au problème de Markowitz ?**

En **théorie moderne du portefeuille (MPT)**, l'objectif est de construire un portefeuille qui :
- **Maximise le rendement espéré** pour un niveau de risque donné, **ou**
- **Minimise le risque** pour un rendement espéré donné.

Cependant, une **contrainte fondamentale** doit être respectée :
> **La somme des poids des actifs dans le portefeuille doit être égale à 1** :
> \[
> \sum_{i=1}^{n} w_i = 1
> \]
Cette contrainte garantit que **100% du capital est alloué** aux actifs (pas de poids négatifs ou supérieurs à 1, sauf si on autorise la vente à découvert).

---

## **🔹 Comparaison entre `optimal1` et `optimal2`**
   Fonction       | Objectif                                  | Contrainte sur \( \sum w_i \) | Adapté à Markowitz ? |
 |----------------|------------------------------------------|-------------------------------|----------------------|
 | **`optimal1`** | Maximiser le rendement pour une **variance fixée** \( c \) | ❌ **Non imposée**            | ❌ Non               |
 | **`optimal2`** | Minimiser la variance pour un **rendement espéré fixé** \( \mu \) | ✅ **Oui, \( \sum w_i = 1 \)** | ✅ **Oui**          |

---

## **📌 Pourquoi `optimal2` est-il plus rigoureux ?**

### **1. `optimal1` : Une solution incomplète**
La fonction `optimal1` calcule un portefeuille qui maximise le rendement pour une variance donnée, **mais elle n'impose pas explicitement** que la somme des poids \( \sum w_i = 1 \).
- **Problème** : Les poids retournés peuvent ne pas sommer à 1, ce qui signifie que le portefeuille n'est pas **pleinement investi** (ou pire, qu'il y a des poids négatifs ou > 1 sans contrôle).
- **Conséquence** : Le portefeuille peut être **non réaliste** ou **non interprétable** en pratique.

---

### **2. `optimal2` : Une solution conforme à Markowitz**
La fonction `optimal2` résout le **vrai problème de Markowitz** :
\[
\min_w w^T V w \quad \text{ sous les contraintes } \quad
\begin{cases}
w^T R = \mu \quad \text{(rendement espéré fixé)} \\
\sum_{i=1}^n w_i = 1 \quad \text{(somme des poids = 1)}
\end{cases}
\]
- **Avantage** :
  - La solution **garantit** que le portefeuille est **pleinement alloué** (\( \sum w_i = 1 \)).
  - Elle est **mathématiquement exacte** pour le problème standard de la frontière efficiente.
  - Elle permet de **tracer la frontière efficiente complète** en faisant varier \( \mu \).

---
## **📊 Illustration**
Pour construire la **frontière efficiente**, on utilise généralement `optimal2` en faisant varier \( \mu \) :
1. Pour chaque valeur de \( \mu \) (ex : de 0.05 à 0.20 par pas de 0.01) :
   - Calculer \( w \) avec `optimal2(R, V, mu)`.
   - Calculer le risque \( \sigma = \sqrt{w^T V w} \).
2. Tracer \( \sigma \) (risque) en fonction de \( \mu \) (rendement).

**→ Résultat** : Une courbe qui représente **tous les portefeuilles optimaux**, avec la garantie que \( \sum w_i = 1 \).

---
## **⚠️ Quand utiliser `optimal1` ?**
`optimal1` peut être utile si :
- On veut **fixer la variance** (et non le rendement).
- On **n’impose pas** la contrainte \( \sum w_i = 1 \) (ce qui est rare en pratique).
- On travaille dans un cadre **théorique** où cette contrainte n’est pas nécessaire.

**Mais en gestion de portefeuille réelle, `optimal2` est la méthode standard.**

---
## **💡 Conclusion**
> **`optimal2` est la fonction qui répond le mieux au problème de Markowitz**, car elle :
> - **Minimise le risque pour un rendement donné**.
> - **Impose \( \sum w_i = 1 \)**, garantissant un portefeuille **valide et réaliste**.
> - Permet de **construire la frontière efficiente complète**.

**→ Pour une implémentation robuste de la MPT, privilégiez `optimal2` !**